In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
import yaml

from src.data.loader import load_dataset
from src.data.windows import make_rolling_windows
from src.features.extractor import extract_window_features

In [3]:
BENCHMARK_CONFIG = PROJECT_ROOT / "configs" / "benchmark.yaml"
TARGET_CONFIG = PROJECT_ROOT / "configs" / "benchmark_targets.yaml"

In [7]:
with open(BENCHMARK_CONFIG, "r", encoding="utf-8") as f:
    benchmark = yaml.safe_load(f)["benchmark"]

with open(TARGET_CONFIG, "r", encoding="utf-8") as f:
    targets = yaml.safe_load(f)["targets"]

In [5]:
TSFM_MODELS = ["chronos2", "moirai2", "timesfm3"]
ALL_MODELS = TSFM_MODELS + ["seasonal_naive"]

In [9]:
benchmark

{'datasets': ['ETTh1',
  'Weather',
  'Electricity',
  'Traffic',
  'Exchange',
  'Solar'],
 'context_length': 512,
 'prediction_lengths': [24, 96, 192],
 'windows_per_target': 20,
 'targets_per_dataset': 3,
 'target_selection_seed': 2026,
 'save_forecasts': False}

In [10]:
targets

{'ETTh1': ['OT', 'HULL', 'MULL'],
 'Weather': ['OT', 'rh (%)', 'SWDR (W/m�)'],
 'Electricity': ['146', '22', '163'],
 'Traffic': ['T683', 'T472', 'T855'],
 'Exchange': ['4', 'OT', '0'],
 'Solar': ['solar_125', 'solar_115', 'solar_27']}

In [11]:
def build_features(benchmark: dict, targets: dict) -> pd.DataFrame:
    rows = []
    for dataset_name in benchmark["datasets"]:
        dataset = load_dataset(dataset_name)
        
        for target in targets[dataset_name]:
            for prediction_length in benchmark["prediction_lengths"]:
                windows = make_rolling_windows(
                    dataset=dataset,
                    context_length=benchmark['context_length'],
                    prediction_length=prediction_length,
                    n_windows=benchmark["windows_per_target"],
                    stride=prediction_length,
                    columns=[target])
                for window in windows:
                    features = extract_window_features(
                        window=window, 
                        seasonal_period=dataset.seasonal_period)
                    rows.append({
                        "dataset": dataset.name,
                        "domain": dataset.domain,
                        "target": target,
                        "window_id": window.window_id,
                        **features})
    return pd.DataFrame(rows)

In [12]:
def load_metrics(benchmark: dict) -> pd.DataFrame:
    metrics_dir = PROJECT_ROOT / "results" / "benchmark" / "metrics"
    
    frames = []
    for dataset_name in benchmark["datasets"]:
        for model in ALL_MODELS:
            path = metrics_dir / f"{model}_{dataset_name}.parquet"
            if not path.exists():
                raise FileNotFoundError(f"Missing benchmark result: {path}")
            frames.append(pd.read_parquet(path))
    return pd.concat(frames, ignore_index=True)

In [ ]:
def build_meta_dataset() -> pd.DataFrame:
    features = build_features(benchmark=benchmark, targets=targets)
    metrics = load_metrics(benchmark=benchmark)
    key = ["dataset", "domain", "target", "window_id", "prediction_length"]
    
    mase = metrics.pivot(index=key, columns="model", values="mase").reset_index()
    mase = mase.rename(columns={
        "chronos2": "chronos2_mase",
        "moirai2": "moirai2_mase",
        "timesfm3": "timesfm3_mase",
        "seasonal_naive": "seasonal_naive_mase"})
    
    meta = features.merge(mase, on=key, how="inner", validate="one_to_one")
    tsfm_column = ["chronos2_mase", "moirai2_mase", "timesfm3_mase"]
    
    model_names = np.array(TSFM_MODELS)
    scores = meta[tsfm_column].to_numpy()
    
    order = np.argsort(scores, axis=1)
    best = np.take_along_axis(scores, order[:, :1], axis=1).ravel()
    second_best = np.take_along_axis(scores, order[:, 1: 2], axis=1).ravel()
    
    meta["best_tsfm"] = model_names[order[:, 0]]
    meta["oracle_mase"] = best
    meta["winner_margin"] = second_best - best
    meta["winner_margin_relative"] = meta["winner_margin"] / np.maximum(best, 1e-8)
    
    return meta

In [16]:
meta = build_meta_dataset()

In [17]:
meta

,dataset,domain,target,window_id,context_length,prediction_length,seasonal_period,horizon_ratio,sampling_interval_seconds,mean,...,missing_rate,outlier_rate,chronos2_mase,moirai2_mase,seasonal_naive_mase,timesfm3_mase,best_tsfm,oracle_mase,winner_margin,winner_margin_relative
0,ETTh1,energy,OT,0,512,24,24,1.000000,3600.0,9.662660,...,0.0,0.015625,0.302008,0.379346,0.888932,0.418249,chronos2,0.302008,0.077338,0.256081
1,ETTh1,energy,OT,1,512,24,24,1.000000,3600.0,9.529934,...,0.0,0.000000,0.725469,0.688865,0.887000,0.772440,moirai2,0.688865,0.036605,0.053138
2,ETTh1,energy,OT,2,512,24,24,1.000000,3600.0,9.385656,...,0.0,0.005859,1.734458,1.762020,2.534228,1.844116,chronos2,1.734458,0.027561,0.015891
3,ETTh1,energy,OT,3,512,24,24,1.000000,3600.0,9.323006,...,0.0,0.009766,0.853528,0.583212,1.864404,0.733799,moirai2,0.583212,0.150587,0.258203
4,ETTh1,energy,OT,4,512,24,24,1.000000,3600.0,9.360652,...,0.0,0.013672,0.557674,0.489419,1.065874,0.589103,moirai2,0.489419,0.068255,0.139461
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1075,Solar,renewable_energy,solar_27,15,512,192,144,1.333333,600.0,2.101758,...,0.0,0.000000,1.775278,1.146685,0.418033,1.094877,timesfm3,1.094877,0.051808,0.047319
1076,Solar,renewable_energy,solar_27,16,512,192,144,1.333333,600.0,2.726172,...,0.0,0.000000,0.451043,0.338515,1.179270,1.236665,moirai2,0.338515,0.112529,0.332419
1077,Solar,renewable_energy,solar_27,17,512,192,144,1.333333,600.0,2.734668,...,0.0,0.000000,0.765953,1.037875,1.047272,0.581152,timesfm3,0.581152,0.184800,0.317989
1078,Solar,renewable_energy,solar_27,18,512,192,144,1.333333,600.0,3.117871,...,0.0,0.000000,0.986650,0.976231,1.073435,0.929872,timesfm3,0.929872,0.046360,0.049856


In [18]:
output_dir = PROJECT_ROOT / "results" / "meta_dataset"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "meta_dataset.parquet"
meta.to_parquet(output_path, index=False)

In [19]:
print(f"Shape: {meta.shape}")

Shape: (1080, 31)


In [20]:
print("Rows by dataset:")
print(meta["dataset"].value_counts().sort_index())

Rows by dataset:
dataset
ETTh1          180
Electricity    180
Exchange       180
Solar          180
Traffic        180
Weather        180
Name: count, dtype: int64


In [21]:
print("Best TSFM:")
print(meta["best_tsfm"].value_counts())

Best TSFM:
best_tsfm
chronos2    407
moirai2     348
timesfm3    325
Name: count, dtype: int64


In [ ]:
print("Mean TSFM MASE")
print(meta[["chronos2_mase", "moirai2_mase", "timesfm3_mase"]].mean().sort_values())

Mean TSFM MASE
moirai2_mase     2.106194
chronos2_mase    2.172830
timesfm3_mase    2.273430
dtype: float64


In [25]:
print(f"Saved to {output_path.relative_to(PROJECT_ROOT).as_posix()}")

Saved to results/meta_dataset/meta_dataset.parquet
